<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [ ]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 MB 3.8 MB/s eta 0:00:00


In [ ]:
#DRIVE
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os

# Params
VERBOSE = True
CHOSEN = 'llama'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
IT = 10
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'
BASE_FOLDER = 'drive/MyDrive/NLP_proj/estimation/'
OVERWRITE = ['']

# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf",
             'temperature': 0.6,
             'n_ctx': 40960,
             'chat_format': "qwen"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': "chatml"},
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=False,
                            force_download=True,
                            enable_thinking=True)


def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

prompts = {}

./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

# Game parameter estimation
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

## Estimating everything at once

In [1]:

prompts['all'] = """You are a board‑game analyst that always explains its reasoning before answering.
For any supplied rule excerpt you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting already existing BGG mechanic(s).
3. Judge the rule density and decision depth. Then assign a complexity score (1‑5).
4. From the number of components in the box and player‑interaction patterns infer the optimal player‑count.
6. Assess the progression of a typical game turn. Based on the complexity of the required actions and how much each turn brings the player closer to the final goal, estimate the game’s average duration in minutes.
Explain your reasoning step by step then output a json object with the fields 'mechanics' (list of strings), 'complexity'(1-5), 'optimal player count', 'duration' that matches the schema below.

JSON schema:
{
  "type": "object",
  "properties": {
    "reasoning": {"type": "string"},
    "answer": {"type": "object", "properties": {
      "mechanics": {"type": "array", "items": {"type": "string"}},
      "complexity": {"type": "number", "minimum": 1, "maximum": 5},
      "optimal player count": {"type": "number"},
      "duration": {"type": "number"},
      "required": ["mechanics", "complexity", "optimal player count", "duration"]
    }}
  },
  "required": ["reasoning", "answer"]
}
"""

all_rf = {"type": "json_object",
          "schema": {
              "type": "object",
              "properties": {
                  "reasoning": {"type": "string"},
                  "answer": {"type": "object", "properties": {
                      "mechanics": {"type": "array", "items": {"type": "string"}},
                      "complexity": {"type": "number", "minimum": 1, "maximum": 5},
                      "optimal player count": {"type": "number"},
                      "duration": {"type": "number"},
                      "required": ["mechanics", "complexity", "optimal player count", "duration"]
                      }
                    }
                  },
              "required": ["reasoning", "answer"]
              }
          }



## Estimating each parameter separately

### Mechanics

In [2]:
prompts['mechanics'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting BGG mechanic(s).

here is a complete list of BGG mechanics:
Acting, Action / Event, Action Drafting, Action Points, Action Queue, Action Retrieval,
Action Timer, Advantage Token, Alliances, Area Majority / Influence, Area Movement, Area-Impulse,
Auction / Bidding, Auction Compensation, Auction: Dexterity, Auction: Dutch, Auction: Dutch Priority,
Auction: English, Auction: Fixed Placement, Auction: Multiple Lot, Auction: Once Around,
Auction: Sealed Bid,Auction: Turn Order Until Pass, Automatic Resource Growth, Betting and Bluffing,
Bias, Bids As Wagers, Bingo, Bribery, Campaign / Battle Card Driven, Card Play Conflict Resolution,
Catch the Leader, Chaining, Chit-Pull System, Closed Drafting, Closed Economy Auction, Command Cards,
Commodity Speculation, Communication Limits, Connections, Constrained Bidding, Contracts,
Cooperative Game, Crayon Rail System, Critical Hits and Failures, Cube Tower, Deck Construction, "Deck, Bag, and Pool Building", Deduction,Delayed Purchase, Dice Rolling, Die Icon Resolution, Different Dice Movement, Drawing, Elapsed Real Time Ending, Enclosure, End Game Bonuses, Events, Finale Ending, Flicking, Follow, Force Commitment, Grid Coverage, Grid Movement, Hand Management, Hexagon Grid, Hidden Movement, Hidden Roles, Hidden Victory Points, Highest-Lowest Scoring, Hot Potato, "I Cut, You Choose", Impulse Movement, Income, Increase Value of Unchosen Resources, Induction, Interrupts, Investment, Kill Steal, King of the Hill, Ladder Climbing, Layering, Legacy Game, Line Drawing, Line of Sight, Loans, Lose a Turn, Mancala,
Map Addition, Map Deformation, Map Reduction, Market, Matching, Measurement Movement,
Melding and Splaying, Memory, Minimap Resolution, Modular Board, Move Through Deck,
Movement Points, Movement Template, Moving Multiple Units, Multi-Use Cards, Multiple Maps,
Narrative Choice / Paragraph, Negotiation, Neighbor Scope, Network and Route Building,
Once-Per-Game Abilities, Open Drafting, Order Counters, Ordering, Ownership, Paper-and-Pencil,
Passed Action Token, Pattern Building, Pattern Movement, Pattern Recognition, Physical Removal,
Pick-up and Deliver, Pieces as Map, Player Elimination, Player Judge, Point to Point Movement,
Predictive Bid, Prisoner's Dilemma, Programmed Movement, Push Your Luck, Questions and Answers, Race,
Random Production, Ratio / Combat Results Table, Re-rolling and Locking, Real-Time, Relative Movement,
Resource Queue, Resource to Move, Rock-Paper-Scissors, Role Playing, Roles with Asymmetric Information,
Roll / Spin and Move, Rondel, Scenario / Mission / Campaign Game, Score-and-Reset Game, Secret Unit Deployment,
Selection Order Bid, Semi-Cooperative Game, Set Collection, Simulation, Simultaneous Action Selection,
Singing, Single Loser Game, Slide / Push, Solo / Solitaire Game, Speed Matching, Spelling, Square Grid,
Stacking and Balancing, Stat Check Resolution, Static Capture, Stock Holding, Storytelling, Sudden Death Ending,
Tags, Take That, Targeted Clues, Team-Based Game, Tech Trees / Tech Tracks, Three Dimensional Movement,
Tile Placement, Track Movement, Trading, Traitor Game, Trick-taking, Tug of War, Turn Order: Auction,
Turn Order: Claim Action, Turn Order: Pass Order, Turn Order: Progressive, Turn Order: Random,
Turn Order: Role Order, Turn Order: Stat-Based, Turn Order: Time Track, Variable Phase Order,
Variable Player Powers, Variable Set-up, Victory Points as a Resource, Voting, Worker Placement,
Worker Placement with Dice Workers, "Worker Placement, Different Worker Types", Zone of Control
"""


### Complexity rating

In [3]:
prompts['complexity'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Analyze Learning Complexity
- Analyze the length of the text, setup steps, rule exceptions and other factor that may indicate how rule intensive the game is.
- Reason about how quickly a new player could grasp the basics.
2. Analyze Playing Complexity
- Look at in‑game actions per turn, resource management, simultaneous moves and number of element to manage
- Estimate mental load during a typical play session.
3. Analyze Strategy/Tactics
- Examine depth of decision space, long‑term planning, branching possibilities, and how impactful is a wrong decision.
4. Convert each qualitative assessment to a numeric rating (1‑5)
- Provide a short justification for each number.
5 Compute the complexity rating
- Average = (Learning + Playing + Strategy) / 3
- Round to one decimal.

**Final Output**
Learning: X
Playing: Y
Strategy: Z
Overall Complexity: W.W"""

### Optimal player count

In [4]:
prompts['player'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Identify the player‑count range stated in the rules (minimum‑maximum). If the rulebook does not explicitly state a player count, infer the appropriate range from the game components and mechanics described in the box contents.
2. Examine how the core mechanics scale with player number
3. Consider the impact on play time, player interaction, and variance (e.g., games that become chaotic with many players or too slow with few).
4. Weigh the pros and cons of each possible player count within the allowed range.
5. find the optimal player count based on your observations

**Final Output**
Optimal player count: X
"""

### Game duration

In [5]:
prompts['duration'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Assess the progression of a typical game turn.
2. Evaluate the complexity of the required actions and how long a turn would last
3. Evaluate how much each turn brings the player closer to the final goal
4. Based on your observation estimate the game’s average duration in minutes.

**Final Output**
Average game duration: X minutes
"""

## Execution

In [ ]:
# select only prompt not executed
to_do = [p for p in prompts.keys if '{CHOSEN}_{prompt}.json' not in os.listdir(BASE_FOLDER)
                                 or '{CHOSEN}_{prompt}.json' in OVERWRITE]
for prompt in to_do:
    output_dict = {g: {it: '' for it in range(IT)} for g in FILE_NAMES}
    for g,it in tqdm([(g,it) for g in FILE_NAMES for it in range(IT)]):
        rulebook = requests.get(BASE_URL +g+'.txt').text
        out = model.create_chat_completion(generate_message(prompts[prompt], rulebook),
                                           temperature=models[CHOSEN]['temperature'],
                                           response_format = all_rf if prompt == 'all' else None,
                                           )['choices'][0]['message']['content']
        output_dict[g][it] = out
        if(VERBOSE):
            print(out)

    with open(f'{BASE_FOLDER}{CHOSEN}_{prompt}.json','w') as f:
        json.dump(dict(output_dict),f)